In [1]:
# =============================================================================
# CELL 1 - Imports
# =============================================================================

from __future__ import annotations

import importlib
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module
import src.pago_pipeline.pca_plot.pca_plot_dataset as pca_plot_dataset_module
import src.pago_pipeline.pca_plot.pca_plot_render as pca_plot_render_module
import src.pago_pipeline.pca_plot.pca_plot_snapshot as pca_plot_snapshot_module
import src.pago_pipeline.pca_kmeans_snapshot as pca_kmeans_snapshot_module
from src.pago_pipeline.storage import sha256_of_file

# Reload pipeline modules so notebook reruns pick up local code changes.
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)
pca_plot_dataset_module = importlib.reload(pca_plot_dataset_module)
pca_plot_render_module = importlib.reload(pca_plot_render_module)
pca_plot_snapshot_module = importlib.reload(pca_plot_snapshot_module)
pca_kmeans_snapshot_module = importlib.reload(pca_kmeans_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
load_latest_pca_kmeans_snapshot = (
    pca_kmeans_snapshot_module.load_latest_pca_kmeans_snapshot
)
resolve_pca_plot_snapshot = (
    pca_plot_snapshot_module.resolve_pca_plot_snapshot
)
latest_pca_plot_snapshot_is_available = (
    pca_plot_snapshot_module.latest_pca_plot_snapshot_is_available
)

In [2]:
# =============================================================================
# CELL 2 - Load environment and resolve project root
# =============================================================================

dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your project configuration at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Programming\Python\pAgo-project


In [3]:
# =============================================================================
# CELL 3 - Define PCA plot snapshot configuration
# =============================================================================

PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "04-analysis" / "pca_kmeans"
)
PCA_PLOT_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "05-visualization" / "pca_plot"
)

PCA_KMEANS_SNAPSHOT_MODE = SnapshotMode.reuse_latest
PCA_PLOT_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
PLOT_DIMENSION_MODE = "auto"
PLOT_ROTATION_DEGREES = 225.0
PLOT_MIRROR_X_AXIS = True
OPEN_HTML_IN_BROWSER = False
UPDATE_LATEST_DIRECTORY = True

print(f"PCA/KMeans snapshot root directory: {PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY}")
print(f"PCA plot output root directory: {PCA_PLOT_SNAPSHOT_ROOT_DIRECTORY}")
print(f"PCA plot snapshot mode: {PCA_PLOT_SNAPSHOT_MODE}")
print(f"Plot dimension mode: {PLOT_DIMENSION_MODE}")
print(f"Rotation degrees: {PLOT_ROTATION_DEGREES}")
print(f"Mirror X axis: {PLOT_MIRROR_X_AXIS}")
print(f"Open HTML in browser: {OPEN_HTML_IN_BROWSER}")

PCA/KMeans snapshot root directory: C:\Programming\Python\pAgo-project\data\04-analysis\pca_kmeans
PCA plot output root directory: C:\Programming\Python\pAgo-project\data\05-visualization\pca_plot
PCA plot snapshot mode: reuse_latest_or_create
Plot dimension mode: auto
Rotation degrees: 225.0
Mirror X axis: True
Open HTML in browser: False


In [4]:
# =============================================================================
# CELL 4 - Resolve active PCA/KMeans snapshot
# =============================================================================

pca_kmeans_snapshot_payload = load_latest_pca_kmeans_snapshot(
    snapshot_root_directory=PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY,
)

pca_kmeans_snapshot_directory = pca_kmeans_snapshot_payload["snapshot_directory"]
pca_kmeans_manifest_file_path = pca_kmeans_snapshot_payload["manifest_file_path"]
pca_kmeans_manifest_payload = pca_kmeans_snapshot_payload["manifest"]
cluster_assignments_file_path = pca_kmeans_snapshot_payload[
    "cluster_assignments_file_path"
]

print("Resolved PCA/KMeans snapshot successfully.")
print(f"Snapshot directory: {pca_kmeans_snapshot_directory}")
print(f"Cluster assignments path: {cluster_assignments_file_path}")

Resolved PCA/KMeans snapshot successfully.
Snapshot directory: C:\Programming\Python\pAgo-project\data\04-analysis\pca_kmeans\latest
Cluster assignments path: C:\Programming\Python\pAgo-project\data\04-analysis\pca_kmeans\latest\cluster_assignments.csv


C:\Programming\Python\pAgo-project\src\pago_pipeline\pca_kmeans_snapshot.py:958: DtypeWarning: Columns (0: gbseq__secondary_accessions__secondary_accn, 1: taxonomy__09, 2: taxonomy__10, 3: reference__consortium, 4: feature__cds__qual__db_xref, 5: feature__cds__qual__gene, 6: feature__cds__qual__gene_synonym, 7: feature__cds__qual__old_locus_tag, 8: feature__gene__interval__accession, 9: feature__gene__location, 10: feature__gene__qual__g_o_function, 11: feature__gene__qual__gene, 12: feature__gene__qual__gene_synonym, 13: feature__gene__qual__locus_tag, 14: feature__het__interval__accession, 15: feature__het__interval__point, 16: feature__het__location, 17: feature__het__qual__heterogen, 18: feature__non_std_res__interval__accession, 19: feature__non_std_res__interval__point, 20: feature__non_std_res__location, 21: feature__non_std_res__qual__non_std_residue, 22: feature__protein__qual__function, 23: feature__protein__qual__g_o_component, 24: feature__protein__qual__name, 25: feature__

In [5]:
# =============================================================================
# CELL 5 - Resolve active PCA plot snapshot
# =============================================================================

pca_plot_snapshot_payload = resolve_pca_plot_snapshot(
    snapshot_mode=PCA_PLOT_SNAPSHOT_MODE,
    snapshot_root_directory=PCA_PLOT_SNAPSHOT_ROOT_DIRECTORY,
    plot_dimension_mode=PLOT_DIMENSION_MODE,
    source_pca_kmeans_snapshot_root_directory=PCA_KMEANS_SNAPSHOT_ROOT_DIRECTORY,
    plot_rotation_degrees=PLOT_ROTATION_DEGREES,
    plot_mirror_x_axis=PLOT_MIRROR_X_AXIS,
    open_html_in_browser=OPEN_HTML_IN_BROWSER,
    update_latest_directory=UPDATE_LATEST_DIRECTORY,
)

pca_plot_snapshot_directory = pca_plot_snapshot_payload["snapshot_directory"]
pca_plot_manifest_file_path = pca_plot_snapshot_payload["manifest_file_path"]
pca_plot_manifest_payload = pca_plot_snapshot_payload["manifest"]
plot_data_file_path = pca_plot_snapshot_payload["plot_data_file_path"]
plot_html_file_path = pca_plot_snapshot_payload["plot_html_file_path"]
plot_profiling_log_file_path = pca_plot_snapshot_payload[
    "profiling_log_file_path"
]
plot_data_dataframe = pca_plot_snapshot_payload["plot_data"]
plot_profiling_log_dataframe = pca_plot_snapshot_payload["profiling_log"]

print("Resolved PCA plot snapshot successfully.")
print(f"Snapshot directory: {pca_plot_snapshot_directory}")
print(f"Plot data path: {plot_data_file_path}")
print(f"Interactive HTML path: {plot_html_file_path}")

C:\Programming\Python\pAgo-project\src\pago_pipeline\pca_kmeans_snapshot.py:958: DtypeWarning: Columns (0: gbseq__secondary_accessions__secondary_accn, 1: taxonomy__09, 2: taxonomy__10, 3: reference__consortium, 4: feature__cds__qual__db_xref, 5: feature__cds__qual__gene, 6: feature__cds__qual__gene_synonym, 7: feature__cds__qual__old_locus_tag, 8: feature__gene__interval__accession, 9: feature__gene__location, 10: feature__gene__qual__g_o_function, 11: feature__gene__qual__gene, 12: feature__gene__qual__gene_synonym, 13: feature__gene__qual__locus_tag, 14: feature__het__interval__accession, 15: feature__het__interval__point, 16: feature__het__location, 17: feature__het__qual__heterogen, 18: feature__non_std_res__interval__accession, 19: feature__non_std_res__interval__point, 20: feature__non_std_res__location, 21: feature__non_std_res__qual__non_std_residue, 22: feature__protein__qual__function, 23: feature__protein__qual__g_o_component, 24: feature__protein__qual__name, 25: feature__

Resolved PCA plot snapshot successfully.
Snapshot directory: C:\Programming\Python\pAgo-project\data\05-visualization\pca_plot\snapshots\2026-04-13T13-56-13Z__q_5b96ea0cbc2f
Plot data path: C:\Programming\Python\pAgo-project\data\05-visualization\pca_plot\snapshots\2026-04-13T13-56-13Z__q_5b96ea0cbc2f\plot_data.csv
Interactive HTML path: C:\Programming\Python\pAgo-project\data\05-visualization\pca_plot\snapshots\2026-04-13T13-56-13Z__q_5b96ea0cbc2f\interactive_plot.html


In [6]:
# =============================================================================
# CELL 6 - Print PCA plot snapshot summary
# =============================================================================

pca_plot_manifest_file_sha256 = sha256_of_file(
    input_file_path=pca_plot_manifest_file_path,
)
plot_data_file_sha256 = sha256_of_file(input_file_path=plot_data_file_path)
plot_html_file_sha256 = sha256_of_file(input_file_path=plot_html_file_path)

print("PCA plot snapshot is ready.")
print(
    f"Snapshot created at UTC: {pca_plot_manifest_payload['snapshot_created_at_utc']}"
)
print(f"Plot data rows: {len(plot_data_dataframe)}")
print(f"Selected PCA component count: {pca_plot_manifest_payload['selected_pca_component_count']}")
print(f"Selected cluster count k: {pca_plot_manifest_payload['selected_cluster_count_k']}")
print(f"Selection reason: {pca_plot_manifest_payload['selection_reason']}")
print(f"Plot data SHA-256: {plot_data_file_sha256}")
print(f"HTML SHA-256: {plot_html_file_sha256}")
print(f"Manifest SHA-256: {pca_plot_manifest_file_sha256}")

PCA plot snapshot is ready.
Snapshot created at UTC: 2026-04-13T13:56:13Z
Plot data rows: 41345
Selected PCA component count: 2
Selected cluster count k: 3
Selection reason: threshold_pass_then_max_silhouette
Plot data SHA-256: 0fe9d2fefa0128d46d9df9e12390a5f98675384eccd8071ea83a5bd450cca666
HTML SHA-256: a259ad8d24c49fc7e4240c6f7f04f0d48ef1947820631cee5ad40ca3c545307f
Manifest SHA-256: ebd5a651f3c06222ea3e842e5044a997c15d77a57ea670abe8c554ae90d85f73


In [7]:
# =============================================================================
# CELL 7 - Inspect plot-ready dataframe and summaries
# =============================================================================

plot_dataframe_preview_row_limit = 10
plot_dataframe_preview = plot_data_dataframe.head(plot_dataframe_preview_row_limit).copy()
rendered_plot_dimension_count = int(
    pca_plot_manifest_payload.get("rendered_plot_dimension_count", 3)
)
plot_coordinate_column_names = ["plot_pc1", "plot_pc2"]
if "plot_pc3" in plot_dataframe_preview.columns:
    plot_coordinate_column_names.append("plot_pc3")

cluster_size_summary_dataframe = (
    plot_data_dataframe["cluster_label"]
    .value_counts()
    .rename_axis("cluster_label")
    .reset_index(name="row_count")
    .sort_values("cluster_label")
    .reset_index(drop=True)
)
pago_summary_dataframe = (
    plot_data_dataframe["pago_label"]
    .value_counts()
    .rename_axis("pago_label")
    .reset_index(name="row_count")
)

plot_preview_column_names = [
    "cluster_label",
    "description_label",
    "organism_label",
    "phylum_label",
    "class_label",
    "family_label",
    "genus_label",
    "pago_label",
] + plot_coordinate_column_names

print(f"Rendered plot dimension: {rendered_plot_dimension_count}D")
print("Plot-ready dataframe preview:")
display(plot_dataframe_preview[plot_preview_column_names])
print("Cluster size summary:")
display(cluster_size_summary_dataframe)
print("pAgo label summary:")
display(pago_summary_dataframe)


Rendered plot dimension: 2D
Plot-ready dataframe preview:


,cluster_label,description_label,organism_label,phylum_label,class_label,family_label,genus_label,pago_label,plot_pc1,plot_pc2
0,0,protein_uid=1000250755|accession=KXK13845.1|le...,Chloroflexi bacterium OLB15,Chloroflexota,Unknown,Unknown,Unknown,pAgo,-0.212879,-0.703975
1,0,protein_uid=1000266463|accession=KXK28958.1|le...,Candidatus Brocadia sinica,Planctomycetota,Candidatus Brocadiia,Candidatus Brocadiaceae,Candidatus Brocadia,pAgo,2.152251,1.020026
2,0,protein_uid=1000285434|accession=KXK47085.1|le...,Bacteroidetes bacterium OLB10,Bacteroidota,Unknown,Unknown,Unknown,pAgo,0.900389,1.923814
3,0,protein_uid=1000285973|accession=KXK47585.1|le...,Bacteroidetes bacterium OLB10,Bacteroidota,Unknown,Unknown,Unknown,pAgo,1.329730,1.663729
4,1,protein_uid=1000287044|accession=KXK48581.1|le...,Chloroflexi bacterium OLB13,Chloroflexota,Unknown,Unknown,Unknown,pAgo,-0.567036,-1.839629
5,0,protein_uid=1000371080|accession=WP_061113836....,Bacillus cereus,Bacillota,Bacilli,Bacillaceae,Bacillus,pAgo,3.384447,1.098967
6,2,protein_uid=1000882329|accession=WP_061139231....,Bacillus,Bacillota,Bacilli,Bacillaceae,Unknown,pAgo,8.399231,-0.270355
7,0,protein_uid=1000882353|accession=WP_061139255....,Bacillus,Bacillota,Bacilli,Bacillaceae,Unknown,pAgo,3.652896,0.964234
8,0,protein_uid=1000934900|accession=WP_061181381....,Pseudomonas aeruginosa,Pseudomonadota,Gammaproteobacteria,Pseudomonadaceae,Pseudomonas,pAgo,-0.059396,0.880626
9,0,protein_uid=1001028003|accession=KXO00930.1|le...,Aequorivita aquimaris,Bacteroidota,Flavobacteriia,Flavobacteriaceae,Aequorivita,pAgo,1.284754,2.075599


Cluster size summary:


,cluster_label,row_count
0,0,33503
1,1,6486
2,2,1356


pAgo label summary:


,pago_label,row_count
0,pAgo,25185
1,Other,16160


In [8]:
# =============================================================================
# CELL 8 - Expose downstream variables
# =============================================================================

print("Variables exposed for downstream notebooks:")
print("- pca_kmeans_snapshot_directory")
print("- pca_kmeans_manifest_payload")
print("- pca_plot_snapshot_directory")
print("- pca_plot_manifest_payload")
print("- plot_data_file_path")
print("- plot_html_file_path")
print("- plot_profiling_log_file_path")
print("- plot_data_dataframe")
print("- plot_profiling_log_dataframe")
print("- cluster_size_summary_dataframe")
print("- pago_summary_dataframe")

Variables exposed for downstream notebooks:
- pca_kmeans_snapshot_directory
- pca_kmeans_manifest_payload
- pca_plot_snapshot_directory
- pca_plot_manifest_payload
- plot_data_file_path
- plot_html_file_path
- plot_profiling_log_file_path
- plot_data_dataframe
- plot_profiling_log_dataframe
- cluster_size_summary_dataframe
- pago_summary_dataframe
